# Navier--Stokes inference with ICL-DM

This notebook follows `IND_NS.py`. It loads the same NS dataset, KME embeddings, and three checkpoints, but runs each stage in a separate cell and displays every figure inline.

In [ ]:
from pathlib import Path
import os
import sys

candidates = [Path.cwd(), Path.cwd().parent]
REPO_ROOT = next((p.resolve() for p in candidates if (p / 'IND_NS.py').exists()), None)
if REPO_ROOT is None:
    raise FileNotFoundError('Run this notebook from ICL_DM_demo/ or ICL_DM_demo/notebooks/.')
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print('Repository root:', REPO_ROOT)

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from config.model_config_vpsde import marginal_prob_fn, get_sde_forward_fn
from utils.utils import make_image_meta
from utils.train_ICL_utils import ode_solver
from utils.metric_utils import w2_pot, mmd_rbf, melr, pdf_kde, jsd_kde
from utils.fid_utils_imagenet import compute_fid_inception

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## Parameters

These defaults are copied from `IND_NS.py`. Change them so that the dataset and checkpoint names match the run you trained.

In [ ]:
K = 10
B = 1000
n = 10
m = 10
Nx = 128
bs = 32
epoch = 2000
num_prompts = 2
class_idx = 0
seed = 42
num_steps = 500
chunk_size = 20

In [ ]:
data_path = REPO_ROOT / 'data' / f'NS_train_K{K}_B{B}.npy'
kme_path = REPO_ROOT / 'data' / 'NS_KMEresults.npz'
icl_path = REPO_ROOT / 'mdls' / f'NS_Gen_K_{K}_B_{B}_n_{n}_np_{num_prompts}_epoch_{epoch}_ckpt.pt'
cond_path = REPO_ROOT / 'mdls' / f'NS_cond_Gen_K_{K}_B_{B}_n_{n}_np_{num_prompts}_epoch_{epoch}_ckpt.pt'
sno_path = REPO_ROOT / 'mdls' / f'NS_SNO_Gen_K_{K}_B_{B}_n_{n}_np_{num_prompts}_epoch_{epoch}_ckpt.pt'

required = {
    'NS dataset': data_path,
    'KME state': kme_path,
    'ICL-DM checkpoint': icl_path,
    'Conditional-DM checkpoint': cond_path,
    'SNO checkpoint': sno_path,
}
for label, path in required.items():
    print(f"{'OK' if path.exists() else 'MISSING':7s}  {label:28s}  {path}")

missing = [str(path) for path in required.values() if not path.exists()]
if missing:
    raise FileNotFoundError('Prepare/train the missing files before inference:\n' + '\n'.join(missing))

## Load the NS dataset and KME embeddings

In [ ]:
c = np.load(data_path).astype(np.float32)
print('Raw NS shape:', c.shape)

if c.ndim == 4:
    c = make_image_meta(c)
    c = c[..., None]

train = torch.from_numpy(c).float()
with np.load(kme_path) as kme_state:
    embeddings = kme_state['train_embeddings'].copy()

print('Model input shape:', tuple(train.shape))
print('KME embedding shape:', embeddings.shape)
print('Value range:', float(train.min()), float(train.max()))

## Select one viscosity class, prompts, references, and shared noise

In [ ]:
if not (0 <= class_idx < train.shape[0]):
    raise IndexError(f'class_idx={class_idx} is outside [0, {train.shape[0] - 1}]')
if m > train.shape[1]:
    raise ValueError(f'm={m} exceeds the {train.shape[1]} samples available in this class')

rng = np.random.default_rng(seed)
prompt_indices = rng.choice(train.shape[1], m, replace=False)
torch_generator = torch.Generator().manual_seed(seed)
reference_indices = torch.randperm(train.shape[1], generator=torch_generator)

prompts = train[class_idx, prompt_indices].to(device)
ref_samples = train[class_idx, reference_indices].to(device)
noise = torch.randn(bs, Nx, Nx, 1, generator=torch_generator).to(device)

print('Prompt indices:', prompt_indices.tolist())
print('Prompts:', tuple(prompts.shape))
print('References:', tuple(ref_samples.shape))
print('Noise:', tuple(noise.shape))

In [ ]:
def _field_array(x):
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().numpy()
    x = np.asarray(x)
    if x.ndim == 4 and x.shape[-1] == 1:
        x = x[..., 0]
    return x

def show_fields(x, title, ncols=5, cmap='jet'):
    x = _field_array(x)
    nshow = min(len(x), ncols)
    fig, axes = plt.subplots(1, nshow, figsize=(3 * nshow, 3), squeeze=False)
    for j in range(nshow):
        axes[0, j].contourf(x[j], levels=36, cmap=cmap)
        axes[0, j].set_axis_off()
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

show_fields(prompts, f'NS prompts: class {class_idx}', ncols=min(5, m))

In [ ]:
show_fields(ref_samples, f'NS reference samples: class {class_idx}', ncols=5)

## Load the three checkpoints

In [ ]:
mdl = torch.load(icl_path, weights_only=False, map_location=device)
mdl_cond = torch.load(cond_path, weights_only=False, map_location=device)
mdl_sno = torch.load(sno_path, weights_only=False, map_location=device)

mdl.eval()
mdl_cond.eval()
mdl_sno.eval()
print('Loaded ICL-DM, Conditional-DM, and SNO.')

In [ ]:
def normalize_generated(x):
    max_abs = x.reshape(x.shape[0], -1).abs().max(dim=1).values.reshape(-1, 1, 1, 1)
    return x / (max_abs + 1e-8)

def run_image_prompt_model(model):
    chunks = []
    for start in range(0, bs, chunk_size):
        chunks.append(
            ode_solver(
                model, marginal_prob_fn, get_sde_forward_fn,
                noise[start:start + chunk_size], prompts,
                forward=2, eps=1e-5, use_em=True, num_steps=num_steps,
            )
        )
    return normalize_generated(torch.cat(chunks, dim=0))

def run_sno_model(model):
    kme_vec = torch.from_numpy(embeddings[class_idx]).float().to(device)
    kme_batch = kme_vec.unsqueeze(0).expand(bs, -1)
    chunks = []
    for start in range(0, bs, chunk_size):
        chunks.append(
            ode_solver(
                model, marginal_prob_fn, get_sde_forward_fn,
                noise[start:start + chunk_size],
                kme_batch[start:start + chunk_size],
                forward=2, eps=1e-5, use_em=True, num_steps=num_steps,
            )
        )
    return normalize_generated(torch.cat(chunks, dim=0))

## ICL-DM inference

In [ ]:
sample_f = run_image_prompt_model(mdl)
print('ICL-DM samples:', tuple(sample_f.shape))

In [ ]:
show_fields(sample_f, 'ICL-DM generated samples', ncols=5)

## Conditional-DM inference

In [ ]:
cond_sample_f = run_image_prompt_model(mdl_cond)
print('Conditional-DM samples:', tuple(cond_sample_f.shape))

In [ ]:
show_fields(cond_sample_f, 'Conditional-DM generated samples', ncols=5)

## SNO inference

In [ ]:
sno_sample_f = run_sno_model(mdl_sno)
print('SNO samples:', tuple(sno_sample_f.shape))

In [ ]:
show_fields(sno_sample_f, 'SNO generated samples', ncols=5)

## Combined sample comparison

In [ ]:
rows = [
    ('Reference', _field_array(ref_samples)),
    ('ICL-DM', _field_array(sample_f)),
    ('Conditional-DM', _field_array(cond_sample_f)),
    ('SNO', _field_array(sno_sample_f)),
]
ncols = min(5, bs)
fig, axes = plt.subplots(len(rows), ncols, figsize=(3 * ncols, 3 * len(rows)))
for row_idx, (label, values) in enumerate(rows):
    for col_idx in range(ncols):
        axes[row_idx, col_idx].contourf(values[col_idx], levels=36, cmap='jet')
        axes[row_idx, col_idx].set_axis_off()
    axes[row_idx, 0].set_title(label, loc='left')
plt.tight_layout()
plt.show()

## Distribution metrics

In [ ]:
ref_np = ref_samples[:bs].detach().cpu().numpy()
sample_np = sample_f.detach().cpu().numpy()
cond_np = cond_sample_f.detach().cpu().numpy()
sno_np = sno_sample_f.detach().cpu().numpy()

def evaluate_method(generated, reference):
    melrw_value, generated_spectrum, reference_spectrum = melr(
        generated, reference, num_modes=12, weights='weighted', return_spectra=True
    )
    return {
        'MMD': mmd_rbf(reference, generated, l=0.01),
        'W2': w2_pot(reference, generated),
        'MELRw': melrw_value,
        'MELRu': melr(generated, reference, num_modes=12, weights='uniform'),
        'JSD': jsd_kde(generated, reference),
    }, generated_spectrum, reference_spectrum

icl_metrics, spectrum_icl, spectrum_ref = evaluate_method(sample_np, ref_np)
cond_metrics, spectrum_cond, _ = evaluate_method(cond_np, ref_np)
sno_metrics, spectrum_sno, _ = evaluate_method(sno_np, ref_np)
metrics = {'ICL-DM': icl_metrics, 'Conditional-DM': cond_metrics, 'SNO': sno_metrics}

headers = ['Method', 'MMD', 'W2', 'MELRw', 'MELRu', 'JSD']
print(f'{headers[0]:16s}' + ''.join(f'{h:>12s}' for h in headers[1:]))
for method, values in metrics.items():
    print(f'{method:16s}' + ''.join(f"{values[h]:12.6f}" for h in headers[1:]))

## Energy-spectrum error

In [ ]:
eps = 1e-8
grid = np.arange(len(spectrum_ref))
fig, ax = plt.subplots(figsize=(6, 4))
for label, spectrum, color in [
    ('ICL-DM', spectrum_icl, 'red'),
    ('Conditional-DM', spectrum_cond, 'green'),
    ('SNO', spectrum_sno, 'purple'),
]:
    error = np.abs(np.log((np.asarray(spectrum) + eps) / (np.asarray(spectrum_ref) + eps)))
    ax.plot(grid[1:], error[1:], linewidth=2, color=color, label=label)
ax.set_xlabel(r'$k$')
ax.set_ylabel(r'$|\log(E_k/E_k^{\mathrm{ref}})|$')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## Scalar-value PDFs

In [ ]:
pdf_grid, pdf_ref = pdf_kde(ref_np)
_, pdf_icl = pdf_kde(sample_np)
_, pdf_cond = pdf_kde(cond_np)
_, pdf_sno = pdf_kde(sno_np)

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(pdf_grid, pdf_ref, color='blue', linewidth=2, label='Reference')
ax.plot(pdf_grid, pdf_icl, color='red', linewidth=2, label='ICL-DM')
ax.plot(pdf_grid, pdf_cond, color='green', linewidth=2, label='Conditional-DM')
ax.plot(pdf_grid, pdf_sno, color='purple', linewidth=2, label='SNO')
ax.set_xlabel(r'$u$')
ax.set_ylabel(r'$PDF(u)$')
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## Optional Inception FID

This is separated because torchvision may download pretrained Inception weights.

In [ ]:
RUN_FID = False
if RUN_FID:
    for method, generated in [
        ('ICL-DM', sample_np),
        ('Conditional-DM', cond_np),
        ('SNO', sno_np),
    ]:
        fid = compute_fid_inception(generated, ref_np, batch_size=32, device=device)
        print(f'{method:16s} FID={fid:.6f}')
else:
    print('FID skipped. Set RUN_FID=True to run it.')